# P-2026-11-17-1: What is the lab's "Services and Utilities" group, and what does it do to grouped ECI?

**Question** (from `proposals/2026-11-14.md` #5, "Services gap"): the grouped ECI in every country report includes an 18th column, *Services and Utilities*. What is actually in that column? How much does it move grouped ECI, coverage, and the Spring 2026 report claims, especially for Singapore, the UK and the UAE?

**Why tonight:** `QUEUE.md` Q01–Q16 are all taken on `main`, and every open `claude/nightly-*` branch is already merged. Under PROTOCOL §1.4, tonight's question comes from `proposals/`. The last three nights all worked on the "essentially unchanged" band, so I took the most report-relevant untouched proposal instead. It turned out to be a data-definition problem, not just a sensitivity check.

- **Date:** 2026-11-17 (America/New_York), nightly run
- **Data:** `data/cleaned/2019–2023clean.csv` (18 groups, re-keyed on ISO3 via `lib/research.lab_groups_iso3`); `data/raw/2019–2023_mega.csv` (HS6); `external-data/raw/wdi_panel.csv` (WDI merchandise exports, GDPpc PPP); `external-data/raw/atlas_hs92_country_year.csv` (Growth Lab published ECI, benchmark only); `code/data-retrieval/datacleaning2025.ipynb` and `Data_Retrieval_2025.ipynb` (read only, for the mapping). No downloads.
- **Years:** 2019 primary; 2020–2022 as checks; 2023 shown but suspect (D1).
- **Peers:** the same dict as the Q02 and 11-14 runs (from each `reports/2026-spring/*.pdf`).
- **Library:** `willbetts/nightly/lib/eci.py` and `lib/research.py`, unchanged.
- **Run time:** a few seconds (printed in the last cell).

## TL;DR

- **"Services and Utilities" is not services.** In every year it is the single HS6 code **999999**, Comtrade's *commodities not specified according to kind* (goods reported without a product code, often for confidentiality). It equals raw 999999 exactly for 148/160 rows in 2019 and for every row in 2022 and 2023 (A1). Including 999999 brings the repo's export totals to WDI *merchandise* (goods) exports: median |log ratio| is **0.0015** with it and **0.0123** without it in 2019 (A2). The intended Comtrade EB02 services pull (in `Data_Retrieval_2025.ipynb`) is not in the data. The UK's column is $17.9B and Singapore's is $21.5B in 2019 (A3).
- **It is the main reason countries drop out of grouped ECI.** Under the drop-any-zero rule, 14–25 countries a year are excluded **only** because this column is zero (25 in 2019). That includes **Thailand in every year** and **the Philippines (a Vietnam peer) in 2019–2021**. No other column zeroes more than 12 countries (B).
- **Dropping the column barely changes ECI and doesn't hurt the income fit.** On the same 120 countries in 2019, Spearman with log GDPpc goes from 0.480 to 0.523. The mean change over 2019–2022 is **+0.004** (and −0.005 for agreement with ATLAS). Ranks stay close (Spearman 0.970–0.977 between the two versions). Among the 18 leave-one-group-out drops, Services is the 10th most influential in 2019 and the 6th in 2022, so it is not special in that respect (C, D).
- **It does touch the report claims.** Dropping it flips the 2019→2022 ECI direction for **UAE** (+0.194 → −0.164) and **Vietnam** (−0.213 → +0.037). It shrinks the UK's rise from +1.029 to +0.212. It also changes the UK's 2022 rank among its peers (2 → 3). In 2022, "Services" is the **2nd-most complex** of the 18 groups (PCI 1.619), and Singapore and the UK have RCA > 1 in it. So SDV and greedy recommendations can point at a column of unidentified goods (E, F).
- **Side finding:** 999999 holds **80.4% of Saudi Arabia's** 2019 exports ($202.4B) and 10.6% of the USA's ($173.9B). Every HS4 analysis drops chapter 99, so it is also missing Saudi Arabia's oil (A3).
- **Recommendation (Will's call):** rename the column, or better, **drop it from the ECI input** and add real services trade separately if wanted (see `data-requests/2026-11-17-services-trade.md`).

## Why this matters for the lab

Every country report runs on the grouped 18-column ECI, and Table 1 of every report describes this group as "Financial, business, legal, communication, travel." This run tests whether one input column means what the report says it means. It also measures what the column costs in coverage (D2: peers vanish) and what it does to the optimizer's target, since the SDV and greedy optimizer can recommend "Services and Utilities" (past reports have done so for China and Malaysia, per `catalog/text/reports/`). The question bears on link 3 of research-agenda §0: does pushing exports into a group move real capabilities?

## Method

**Pre-stated before running:**
- *Identity checks:* (A1) the Services column vs the raw HS6 chapter-99 codes, per country-year. (A2) Raw totals with and without 999999 vs WDI merchandise exports.
- *Coverage:* countries the lab's drop-any-zero rule removes **only** because Services is zero.
- *Primary metric:* Spearman of grouped ECI with log GDPpc (PPP, WDI) on the **same** countries (the baseline's sample), 2019 primary.
- *Secondary metrics:* Spearman with ATLAS published ECI and with oriented repo HS4 ECI. Rank agreement and mean |rank move| between versions.
- *Baseline:* **V0**, the lab's grouped ECI: 18 columns, drop-any-zero, strict RCA > 1, with sorted eigenvalues and the sign oriented to diversity. This is the sign-fixed version, not the notebook's raw sign (Q01).

**Variants (all reported):**
- **V1:** drop the Services column, same rules.
- **R0:** the 18 groups rebuilt from `data/raw` with the lab mapping, as a reproduction reference.
- **V2:** the raw rebuild with code 99 removed **everywhere**, i.e. from Services *and* from the High-complexity aggregate.
- **Context (D):** leave-one-group-out for all 18 columns in 2019 and 2022.

I did **not** pre-state a decision threshold for "drop the column". The recommendation below is a judgment, not a test.

In [1]:
import sys, time, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
import eci, research as R
warnings.filterwarnings('ignore')
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 30); pd.set_option('display.precision', 4)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]
SVC, AGG = 'Services and Utilities', 'High-complexity Products'
REPORT = ['CAN', 'DEU', 'POL', 'SGP', 'ARE', 'GBR', 'VNM']
PEERS = {  # from the 'peer economies (...)' sentence of each reports/2026-spring/*.pdf (same dict as the Q02 / 11-14 runs)
    'CAN': ['USA', 'AUS', 'GBR'], 'DEU': ['FRA', 'GBR', 'ITA'], 'POL': ['DEU', 'CZE', 'HUN'],
    'SGP': ['KOR', 'JPN', 'DEU'], 'ARE': ['SAU', 'KWT', 'QAT'], 'GBR': ['DEU', 'FRA', 'ITA'],
    'VNM': ['SGP', 'MYS', 'PHL']}
FOCUS = sorted(set(REPORT) | {p for v in PEERS.values() for p in v})
print('lib:', eci.__file__); print('focus countries:', len(FOCUS), FOCUS)

lib: /home/user/eci-research/willbetts/nightly/runs/2026-11-17_P-2026-11-17-1-services-column/../../lib/eci.py
focus countries: 20 ['ARE', 'AUS', 'CAN', 'CZE', 'DEU', 'FRA', 'GBR', 'HUN', 'ITA', 'JPN', 'KOR', 'KWT', 'MYS', 'PHL', 'POL', 'QAT', 'SAU', 'SGP', 'USA', 'VNM']


## Results

### A. What is in the column?

In [2]:
def raw_codes_iso3(year, codes=None, chapter=None):
    raw = R.raw_hs6(year)
    if codes is not None:
        raw = raw.loc[raw.index.isin(codes)]
    if chapter is not None:
        raw = raw.loc[raw.index.str[:2] == chapter]
    s = raw.sum(axis=0)
    s = s.drop(index=[c for c in s.index if c in eci.M49_AGGREGATES])
    s.index = [eci.m49_to_iso3(c) for c in s.index]
    return s[pd.notna(s.index)].groupby(level=0).sum()

rows = []
for y in YEARS:
    raw = R.raw_hs6(y)
    ch99 = sorted(raw.index[raw.index.str[:2] == '99'])
    G = R.lab_groups_iso3(y)
    S = G[SVC]
    c = raw_codes_iso3(y, codes=['999999']).reindex(S.index)
    both = c.notna()
    rel = (S[both] - c[both]).abs() / S[both].where(S[both] > 0)
    rows.append({'year': y, 'HS6 codes in ch.99 (raw)': ', '.join(ch99), 'countries': len(S),
                 'Services == 0': int((S <= 0).sum()),
                 'exact match to 999999': int(((S[both] - c[both]).abs() < 1).sum()),
                 'within 1%': int(((rel <= 0.01) | ((S[both] <= 0) & (c[both] <= 0))).sum()),
                 'compared': int(both.sum()),
                 'Services share of world (excl. aggregate)': S.sum() / G.drop(columns=AGG).to_numpy().sum()})
A1 = pd.DataFrame(rows).set_index('year'); A1

,HS6 codes in ch.99 (raw),countries,Services == 0,exact match to 999999,within 1%,compared,Services share of world (excl. aggregate)
year,,,,,,,
2019,999999,160,30,148,153,160,0.0543
2020,999999,154,22,130,137,154,0.0489
2021,999999,136,28,107,119,136,0.0395
2022,999999,141,18,141,141,141,0.0439
2023,999999,154,21,154,154,154,0.0438


**Takeaway (A1):** chapter 99 in the raw pull contains only one HS6 code, `999999`. The Services column equals it exactly for 148/160 rows (2019), 130/154 (2020), 107/136 (2021), and all rows in 2022 and 2023. It is 3.9–5.4% of world exports (excluding the aggregate).

In [3]:
rows = []
for y in YEARS:
    G = R.lab_groups_iso3(y); S = G[SVC]
    c = raw_codes_iso3(y, codes=['999999']).reindex(S.index)
    diff = (S - c)
    bad = diff[(diff.abs() >= 1) | c.isna()]
    rows.append({'year': y, 'rows differing': len(bad),
                 'Services > 999999': int((bad > 0).sum()), 'Services < 999999': int((bad < 0).sum()),
                 'no raw match': int(c.isna().sum()),
                 'median ratio Services/999999 (differing rows)': float((S[bad.index] / c[bad.index]).median()),
                 'sum of differences $B': float(diff.fillna(0).sum() / 1e9)})
A1b = pd.DataFrame(rows).set_index('year'); print(A1b.round(3).to_string())
y = 2021; G = R.lab_groups_iso3(y); c = raw_codes_iso3(y, codes=['999999']).reindex(G.index)
d = (G[SVC] - c); d = d[d.abs() >= 1].sort_values(key=abs, ascending=False)
print('\n2021, largest differences (Services column minus raw 999999, $M):'); print((d.head(8) / 1e6).round(1).to_string())

      rows differing  Services > 999999  Services < 999999  no raw match  median ratio Services/999999 (differing rows)  sum of differences $B
year                                                                                                                                          
2019              12                  4                  8             0                                          1.000                 39.720
2020              24                 14                 10             0                                          1.001                 20.027
2021              29                 17                 12             0                                          1.001                  5.204
2022               0                  0                  0             0                                            NaN                  0.000
2023               0                  0                  0             0                                            NaN                  0.000

**Takeaway (A1b):** the rows that differ in 2019–2021 are the same as raw 999999 in the median (ratio 1.000–1.001). The largest 2021 difference is New Zealand at $1,651.8M. That is a small revision gap, not a services series added on top. The cleaned 2019–2021 files were built from a slightly different raw pull than the one in `data/raw` (the R0 check in C0 also shows only 141/160 rows of 2019 matching on all columns, against 141/141 in 2022).

In [4]:
w = R.wdi()
rows = []
for y in [2019, 2020, 2021, 2022]:
    goods = R.raw_hs6(y)
    tot_all = raw_codes_iso3(y, chapter=None)
    c999 = raw_codes_iso3(y, codes=['999999'])
    merch = w.loc[w.year == y].set_index('iso3')['merch_exports_usd'].dropna()
    idx = tot_all.index.intersection(merch.index)
    idx = [i for i in idx if tot_all[i] >= 1e9]
    r_incl = np.log(tot_all[idx] / merch[idx])
    r_excl = np.log((tot_all[idx] - c999.reindex(idx).fillna(0)) / merch[idx])
    rows.append({'year': y, 'n (exports >= $1B)': len(idx),
                 'median |log(raw/WDI merch)| incl. 999999': r_incl.abs().median(),
                 'median |log(raw/WDI merch)| excl. 999999': r_excl.abs().median(),
                 'share of countries closer incl. 999999': float((r_incl.abs() < r_excl.abs()).mean())})
A2 = pd.DataFrame(rows).set_index('year'); A2

,n (exports >= $1B),median |log(raw/WDI merch)| incl. 999999,median |log(raw/WDI merch)| excl. 999999,share of countries closer incl. 999999
year,,,,
2019,132,0.0015,0.0123,0.6970
2020,129,0.0031,0.0145,0.6744
2021,134,0.0025,0.0154,0.6194
2022,119,0.0033,0.0156,0.7395


**Takeaway (A2):** with 999999 included, the repo's country totals match WDI *merchandise* exports almost exactly (median |log ratio| 0.0015–0.0033 in 2019–2022). Without it they are further off (0.0123–0.0156). 62–74% of countries are closer with it. So 999999 is part of **goods** trade. Real services would push the totals well above merchandise exports.

In [5]:
y = 2019
tot = raw_codes_iso3(y); c999 = raw_codes_iso3(y, codes=['999999'])
sh = (c999 / tot).rename('999999 share of own exports')
big = tot[tot >= 1e9].index
top = pd.concat([sh, (c999 / 1e9).rename('999999 $B'), (tot / 1e9).rename('total $B')], axis=1).loc[big].sort_values('999999 share of own exports', ascending=False)
print('2019, reporters >= $1B:', len(big), '| median 999999 share:', round(sh[big].median(), 4),
      '| reporters with 999999 == 0:', int((c999.reindex(big).fillna(0) <= 0).sum()))
print('\nTop 12 by share:'); print(top.head(12).round(4).to_string())
print('\nSpring 2026 report countries:'); print(top.reindex(REPORT).round(4).to_string())

2019, reporters >= $1B: 134 | median 999999 share: 0.0019 | reporters with 999999 == 0: 18

Top 12 by share:
     999999 share of own exports  999999 $B   total $B
SAU                       0.8037   202.3822   251.8000
ALB                       0.5333     1.4503     2.7194
SUR                       0.2575     0.5550     2.1551
MAC                       0.2360     0.2417     1.0240
ARG                       0.1926    12.5439    65.1141
QAT                       0.1319     9.6213    72.9350
RUS                       0.1304    55.6427   426.7203
DNK                       0.1295    14.2304   109.8791
USA                       0.1058   173.9294  1644.2762
UZB                       0.0945     1.4101    14.9295
FIN                       0.0801     5.8355    72.8392
MEX                       0.0790    36.4036   460.6037

Spring 2026 report countries:
     999999 share of own exports  999999 $B   total $B
CAN                       0.0490    21.8300   445.4927
DEU                       0.0386   

**Takeaway (A3):** 999999 is small for most reporters (median share 0.0019 in 2019, and 18 of 134 reporters ≥ $1B have zero). It is huge for a few: **Saudi Arabia reports 80.4% of its exports ($202.4B) as 999999**, along with Albania (53.3%), Russia (13.0%), Denmark (12.95%) and the USA (10.6%, $173.9B). For the report countries it ranges from 0.02% (UAE) to 5.5% (Singapore). It measures how much a customs authority withholds, not services capability.

### B. What does it cost in coverage?

In [6]:
rows = []; dropped_only_svc = {}
for y in YEARS:
    G = R.lab_groups_iso3(y)
    zero = G <= 0
    kept18 = (~zero).all(axis=1)
    kept_nosvc = (~zero.drop(columns=SVC)).all(axis=1)
    only = list(G.index[kept_nosvc & ~kept18])
    dropped_only_svc[y] = only
    rows.append({'year': y, 'rows': len(G), 'kept (lab rule, 18 cols)': int(kept18.sum()),
                 'kept if Services column dropped': int(kept_nosvc.sum()),
                 'dropped only because Services == 0': len(only),
                 'zeros in Services': int(zero[SVC].sum()),
                 'zeros in next-worst column': int(zero.drop(columns=SVC).sum().max()),
                 'next-worst column': zero.drop(columns=SVC).sum().idxmax()})
B = pd.DataFrame(rows).set_index('year'); print(B.to_string())
for y in YEARS:
    print(y, 'dropped only for Services:', ' '.join(dropped_only_svc[y]))
print('\nFocus countries dropped only because of Services:')
for y in YEARS:
    print(' ', y, [c for c in dropped_only_svc[y] if c in FOCUS])

      rows  kept (lab rule, 18 cols)  kept if Services column dropped  dropped only because Services == 0  zeros in Services  zeros in next-worst column next-worst column
year                                                                                                                                                                      
2019   160                       120                              145                                  25                 30                          12         Chemicals
2020   154                       120                              140                                  20                 22                           8           Weapons
2021   136                       105                              129                                  24                 28                           4         Chemicals
2022   141                       120                              134                                  14                 18                     

**Takeaway (B):** Services is the column that zeroes the most countries in every year (18–30 against 4–12 for the next-worst). The drop-any-zero rule removes **25 / 20 / 24 / 14 / 17 countries (2019–2023) only because of it**, including Thailand every year and the Philippines, a Vietnam peer, in 2019–2021. The data being "missing" here just means a customs office had no unspecified goods to report.

In [7]:
for y, c in [(2019, 'QAT'), (2021, 'KWT'), (2022, 'SAU'), (2023, 'QAT')]:
    G = R.lab_groups_iso3(y)
    if c in G.index:
        print(y, c, 'zero columns:', list(G.columns[(G.loc[c] <= 0)]))
    else:
        print(y, c, 'not in cleaned CSV')

2019 QAT zero columns: ['Chemicals', 'Textile', 'Machinery and Construction']
2021 KWT not in cleaned CSV
2022 SAU not in cleaned CSV
2023 QAT zero columns: ['Weapons']


**Takeaway (B2):** the other gaps among the report peers have other causes. Qatar is zero in Chemicals, Textile and Machinery in 2019 (Weapons in 2023), and Kuwait (2021) and Saudi Arabia (2022) are missing from the cleaned CSV entirely. Dropping the Services column doesn't bring them back.

### C. Does dropping it change ECI or its fit?

In [8]:
def hs4_eci(y):
    return eci.eci_from_exports(R.repo_hs(y))

ATL = R.atlas_cy()
def atlas_eci(y):
    return ATL.loc[ATL.year == y].set_index('iso3')['eci'].dropna()

def lab_eci(X):
    """Lab rules: drop-any-zero, strict RCA (>1), sorted eigenvalues, sign oriented to diversity."""
    return eci.eci_from_exports(eci.notebook_filter(X), strict=True)

MAP_NO_SVC = {g: [c for c in chs if not (g == AGG and c == 99)] for g, chs in R.LAB_GROUPS.items() if g != SVC}

def variants(y):
    G = R.lab_groups_iso3(y)
    Rraw = R.groups_from_raw(y, R.LAB_GROUPS)
    return {
        'V0 lab (18 cols)': lab_eci(G),
        'V1 drop Services col': lab_eci(G.drop(columns=SVC)),
        'R0 rebuilt from raw (18 cols)': lab_eci(Rraw),
        'V2 rebuilt, no 99 anywhere': lab_eci(R.groups_from_raw(y, MAP_NO_SVC)),
    }

# reproduction check: is the raw rebuild (R0) the same matrix as the cleaned CSV?
for y in [2019, 2022]:
    G = R.lab_groups_iso3(y); Rr = R.groups_from_raw(y, R.LAB_GROUPS)
    idx = G.index.intersection(Rr.index)
    rel = ((G.loc[idx] - Rr.loc[idx][G.columns]).abs() / G.loc[idx].where(G.loc[idx] > 0)).max(axis=1)
    print(y, 'cleaned rows', len(G), 'raw rows', len(Rr), 'common', len(idx),
          '| rows with max rel diff < 1e-6:', int((rel.fillna(0) < 1e-6).sum()),
          '| only in cleaned:', sorted(set(G.index) - set(Rr.index))[:8], '| only in raw:', len(set(Rr.index) - set(G.index)))

2019 cleaned rows 160 raw rows 167 common 160 | rows with max rel diff < 1e-6: 141 | only in cleaned: [] | only in raw: 7


2022 cleaned rows 141 raw rows 141 common 141 | rows with max rel diff < 1e-6: 141 | only in cleaned: [] | only in raw: 0


**Takeaway (C0):** the raw rebuild (R0) matches the cleaned CSV row for row in 2022 (141/141) but not in 2019 (141/160 rows; 7 extra raw rows), so R0 and V0 differ in 2019–2021. V2 is therefore compared with R0, not V0.

In [9]:
def metrics(e, y, ref_sample=None):
    if ref_sample is not None:
        e = e.reindex(ref_sample).dropna()
    gpc = np.log(R.gdppc(y, 'ppp'))
    out = {'n': len(e)}
    out['rho logGDPpc'], out['n inc'] = R.spearman(e, gpc)
    out['rho ATLAS'], _ = R.spearman(e, atlas_eci(y))
    out['rho HS4'], _ = R.spearman(e, HS4[y])
    return out

HS4 = {y: hs4_eci(y) for y in YEARS}
VAR = {y: variants(y) for y in YEARS}
rows = []
for y in YEARS:
    V = VAR[y]
    common = V['V0 lab (18 cols)'].index.intersection(V['V1 drop Services col'].index)
    for k, e in V.items():
        rows.append({'year': y, 'variant': k, 'sample': 'own', **metrics(e, y)})
    for k in ['V0 lab (18 cols)', 'V1 drop Services col']:
        rows.append({'year': y, 'variant': k, 'sample': 'V0 sample', **metrics(V[k], y, common)})
    rows.append({'year': y, 'variant': 'HS4 (oriented, reference)', 'sample': 'own', **metrics(HS4[y], y)})
C = pd.DataFrame(rows)
print(C.round(3).to_string(index=False))

 year                       variant    sample   n  rho logGDPpc  n inc  rho ATLAS  rho HS4
 2019              V0 lab (18 cols)       own 120         0.480    118      0.669    0.735
 2019          V1 drop Services col       own 145         0.518    143      0.683    0.748
 2019 R0 rebuilt from raw (18 cols)       own 125         0.527    123      0.694    0.750
 2019    V2 rebuilt, no 99 anywhere       own 151         0.504    149      0.682    0.734
 2019              V0 lab (18 cols) V0 sample 120         0.480    118      0.669    0.735
 2019          V1 drop Services col V0 sample 120         0.523    118      0.692    0.755
 2019     HS4 (oriented, reference)       own 133         0.781    132      0.972    1.000
 2020              V0 lab (18 cols)       own 120         0.534    117      0.718    0.781
 2020          V1 drop Services col       own 140         0.502    137      0.675    0.752
 2020 R0 rebuilt from raw (18 cols)       own 127         0.563    124      0.714    0.783

**Takeaway (C1):** every grouped variant sits far below HS4 (Spearman with log GDPpc 0.48–0.56 grouped against 0.77–0.81 for HS4). That gap is the known granularity problem (Q04), and the Services column is not part of it.

In [10]:
piv = C.pivot_table(index='year', columns=['variant', 'sample'], values='rho logGDPpc')
D = pd.DataFrame({
    'V1-V0 own samples': piv[('V1 drop Services col', 'own')] - piv[('V0 lab (18 cols)', 'own')],
    'V1-V0 same (V0) sample': piv[('V1 drop Services col', 'V0 sample')] - piv[('V0 lab (18 cols)', 'V0 sample')],
    'V2-R0 own samples': piv[('V2 rebuilt, no 99 anywhere', 'own')] - piv[('R0 rebuilt from raw (18 cols)', 'own')],
})
pivA = C.pivot_table(index='year', columns=['variant', 'sample'], values='rho ATLAS')
D['ATLAS: V1-V0 own'] = pivA[('V1 drop Services col', 'own')] - pivA[('V0 lab (18 cols)', 'own')]
D['ATLAS: V1-V0 same sample'] = pivA[('V1 drop Services col', 'V0 sample')] - pivA[('V0 lab (18 cols)', 'V0 sample')]
print('Change in Spearman with log GDPpc (top three columns) and with ATLAS ECI (last two):')
print(D.round(3).to_string())
print('\nMean over 2019-2022:'); print(D.loc[2019:2022].mean().round(3).to_string())

Change in Spearman with log GDPpc (top three columns) and with ATLAS ECI (last two):
      V1-V0 own samples  V1-V0 same (V0) sample  V2-R0 own samples  ATLAS: V1-V0 own  ATLAS: V1-V0 same sample
year                                                                                                          
2019              0.037                   0.042             -0.023             0.014                     0.022
2020             -0.031                  -0.025             -0.062            -0.043                    -0.026
2021              0.015                   0.004             -0.016            -0.029                    -0.028
2022             -0.006                  -0.005             -0.067            -0.030                     0.013
2023             -0.043                  -0.045             -0.036            -0.045                    -0.027

Mean over 2019-2022:
V1-V0 own samples           0.004
V1-V0 same (V0) sample      0.004
V2-R0 own samples          -0.042
ATLAS: V1-V0 o

**Takeaway (C2):** on the same countries, dropping the column changes Spearman with income by +0.042, −0.025, +0.004 and −0.005 in 2019–2022 (mean **+0.004**; −0.045 in the suspect 2023). Agreement with ATLAS changes by a mean of −0.005. That is noise-sized either way. V2, which also removes 99 from the aggregate, is worse than R0 in every year (mean −0.042). Code 99 inside the aggregate carries some income signal, possibly because some large high-income reporters (USA, Denmark, Finland) have big 999999 shares. That is a hypothesis, not tested here (avenue 4).

In [11]:
rows = []
for y in YEARS:
    a, b = VAR[y]['V0 lab (18 cols)'], VAR[y]['V1 drop Services col']
    idx = a.index.intersection(b.index)
    ra, rb = a[idx].rank(ascending=False), b[idx].rank(ascending=False)
    mv = (ra - rb).abs()
    rows.append({'year': y, 'n common': len(idx), 'Spearman V0 vs V1': stats.spearmanr(a[idx], b[idx])[0],
                 'mean |rank move|': mv.mean(), 'share moving >10 ranks': (mv > 10).mean(), 'max move': mv.max(),
                 'max mover': mv.idxmax()})
C3 = pd.DataFrame(rows).set_index('year'); C3

,n common,Spearman V0 vs V1,mean |rank move|,share moving >10 ranks,max move,max mover
year,,,,,,
2019,120,0.9704,5.5000,0.1500,45.0,COG
2020,120,0.9768,4.9167,0.0917,33.0,ZMB
2021,105,0.9774,4.5524,0.0857,29.0,ABW
2022,120,0.9766,4.5667,0.0917,44.0,PYF
2023,122,0.9755,4.6393,0.0984,36.0,ABW


**Takeaway (C3):** V0 and V1 rank the common countries almost identically (Spearman 0.970–0.977). The mean move is 4.6–5.5 ranks, and 8.6–15.0% of countries move more than 10 ranks.

### D. Is Services unusual among the 18 columns?

In [12]:
def logo(y):
    G = R.lab_groups_iso3(y)
    base = lab_eci(G)
    gpc = np.log(R.gdppc(y, 'ppp'))
    b_rho, _ = R.spearman(base, gpc)
    out = []
    for g in G.columns:
        e = lab_eci(G.drop(columns=g))
        idx = base.index.intersection(e.index)
        rho_same, _ = R.spearman(e.reindex(idx), gpc)
        rho_base_same, _ = R.spearman(base.reindex(idx), gpc)
        rho_own, n_own = R.spearman(e, gpc)
        out.append({'dropped group': g, 'countries in sample': len(e), 'gained vs base': len(e) - len(base),
                    'Spearman with base ECI': stats.spearmanr(base[idx], e[idx])[0],
                    'mean |rank move|': (base[idx].rank() - e[idx].rank()).abs().mean(),
                    'd rho income (same sample)': rho_same - rho_base_same,
                    'd rho income (own sample)': rho_own - b_rho})
    return pd.DataFrame(out).set_index('dropped group').sort_values('Spearman with base ECI'), b_rho

LOGO = {}
for y in [2019, 2022]:
    LOGO[y], brho = logo(y)
    print(f'\n{y}: base Spearman with log GDPpc = {brho:.3f}  (n countries in base = {len(VAR[y]["V0 lab (18 cols)"])})')
    print(LOGO[y].round(3).to_string())
    print('Services rank by Spearman-with-base (1 = moves ECI most):', list(LOGO[y].index).index(SVC) + 1, 'of', len(LOGO[y]))


2019: base Spearman with log GDPpc = 0.480  (n countries in base = 120)
                               countries in sample  gained vs base  Spearman with base ECI  mean |rank move|  d rho income (same sample)  d rho income (own sample)
dropped group                                                                                                                                                      
Energy Drilling and Mining                     120               0                   0.819            14.608                       0.184                      0.184
Precious Stones                                120               0                   0.904             9.392                       0.022                      0.022
High-complexity Products                       120               0                   0.921             9.817                      -0.008                     -0.008
Transportation                                 120               0                   0.933             8.51

**Takeaway (D):** no. Dropping Services ranks 10th of 18 for how much ECI moves in 2019 (Spearman 0.970 with the base) and 6th in 2022 (0.977). Energy Drilling and Mining matters far more (0.819 in 2019). Services is unusual only in **coverage**: it is the only drop that gains a meaningful number of countries (+25 in 2019, +14 in 2022). **Odd case:** dropping Energy Drilling and Mining in 2022 gives an ECI that, even oriented to diversity, has Spearman −0.736 with the base. Its income correlation changes by −1.259, so the result turns negative. The 2nd eigenvector at 17 columns picks out a different axis that year (see avenues).

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=False)
for ax, y in zip(axes, [2019, 2022]):
    L = LOGO[y].sort_values('d rho income (same sample)')
    cols = ['#c0392b' if g == SVC else ('#7f8c8d' if g == AGG else '#2c7fb8') for g in L.index]
    ax.barh(range(len(L)), L['d rho income (same sample)'], color=cols)
    ax.set_yticks(range(len(L))); ax.set_yticklabels(L.index, fontsize=7)
    ax.axvline(0, color='k', lw=0.6)
    ax.set_title(f'{y}: change in Spearman(ECI, log GDPpc)\nwhen one group is dropped (same countries)', fontsize=9)
plt.tight_layout(); plt.savefig(FIG / 'leave_one_group_out.png', dpi=90); plt.close()
print('saved figures/leave_one_group_out.png')

saved figures/leave_one_group_out.png


![leave one group out](figures/leave_one_group_out.png)

### E. What happens to the Spring 2026 report claims?

In [14]:
rows = []
for c in FOCUS:
    r = {'iso3': c, 'role': 'report' if c in REPORT else 'peer'}
    for k, lab in [('V0 lab (18 cols)', 'V0'), ('V1 drop Services col', 'V1')]:
        for y in [2019, 2022]:
            e = VAR[y][k]
            r[f'{lab} {y}'] = e.get(c, np.nan)
            r[f'{lab} rank {y}'] = e.rank(ascending=False).get(c, np.nan)
            r[f'{lab} n {y}'] = len(e)
    rows.append(r)
E = pd.DataFrame(rows).set_index('iso3')
E['V0 d'] = E['V0 2022'] - E['V0 2019']; E['V1 d'] = E['V1 2022'] - E['V1 2019']
E['direction flips'] = np.sign(E['V0 d']) != np.sign(E['V1 d'])
show = E[['role', 'V0 2019', 'V1 2019', 'V0 rank 2019', 'V1 rank 2019', 'V0 d', 'V1 d', 'direction flips']]
print(show.round(3).to_string())
print('\nn countries: V0', int(E['V0 n 2019'].iloc[0]), int(E['V0 n 2022'].iloc[0]), '| V1', int(E['V1 n 2019'].iloc[0]), int(E['V1 n 2022'].iloc[0]))
print('Report countries whose 2019->2022 direction flips V0 -> V1:', list(E.index[(E.role == 'report') & E['direction flips']]))
print('Focus countries missing from V0 in any year:', {y: [c for c in FOCUS if c not in VAR[y]['V0 lab (18 cols)'].index] for y in YEARS})
print('Focus countries missing from V1 in any year:', {y: [c for c in FOCUS if c not in VAR[y]['V1 drop Services col'].index] for y in YEARS})

        role  V0 2019  V1 2019  V0 rank 2019  V1 rank 2019   V0 d   V1 d  direction flips
iso3                                                                                     
ARE   report   -2.051   -1.788         118.0         144.0  0.194 -0.164             True
AUS     peer   -1.169   -1.471         110.0         141.0 -0.100  0.191             True
CAN   report   -1.058   -0.891         108.0         113.0  0.571  0.578            False
CZE     peer    1.881    1.890           6.0           8.0 -0.315 -0.026            False
DEU   report    2.031    2.103           1.0           3.0 -0.388 -0.171            False
FRA     peer    0.986    0.994          23.0          30.0 -0.509 -0.246            False
GBR   report    0.461    0.694          39.0          34.0  1.029  0.212            False
HUN     peer    1.941    1.973           4.0           6.0  0.083 -0.052             True
ITA     peer    1.137    1.222          17.0          23.0 -0.257 -0.203            False
JPN     pe

**Takeaway (E):** dropping the column flips the 2019→2022 ECI direction for **2 of 7 report countries**: UAE (+0.194 → −0.164) and Vietnam (−0.213 → +0.037). Peers Australia, Hungary and Malaysia flip too. (The `True` for PHL, QAT and SAU in the table comes from missing values, not real flips.) The UK keeps "increased", but its change falls from **+1.029 to +0.212**. The UK has RCA > 1 in Services in 2021–2022 (F), which fits the idea that much of its grouped rise came through this column. The Philippines enters every year (rank 42 in 2019). Saudi Arabia and Qatar stay missing in some years for other reasons (B2).

In [15]:
rows = []
for c in REPORT:
    r = {'country': c}
    for lab in ['V0', 'V1']:
        own = E.loc[c, f'{lab} d']
        peers = E.loc[PEERS[c], f'{lab} d']
        r[f'{lab} peers with data'] = int(peers.notna().sum())
        r[f'{lab} above peer avg change'] = bool(own > peers.mean()) if peers.notna().any() and pd.notna(own) else None
        lv = E.loc[[c] + PEERS[c], f'{lab} 2022'].dropna()
        r[f'{lab} 2022 rank among peers'] = int(lv.rank(ascending=False)[c]) if c in lv.index else None
    rows.append(r)
E2 = pd.DataFrame(rows).set_index('country')
E2['standing flips'] = E2['V0 above peer avg change'] != E2['V1 above peer avg change']
E2['level rank changes'] = E2['V0 2022 rank among peers'] != E2['V1 2022 rank among peers']
print(E2.to_string())

         V0 peers with data  V0 above peer avg change  V0 2022 rank among peers  V1 peers with data  V1 above peer avg change  V1 2022 rank among peers  standing flips  level rank changes
country                                                                                                                                                                                    
CAN                       3                      True                         3                   3                      True                         3           False               False
DEU                       3                     False                         1                   3                     False                         1           False               False
POL                       3                     False                         4                   3                     False                         4           False               False
SGP                       3                      True       

**Takeaway (E2):** peer-average standing doesn't flip for any of the 7. The UK's 2022 rank among its peers changes (2 → 3). The UAE has only 1 peer with data under both versions, because Saudi Arabia is missing in 2022 and Qatar in 2019.

### F. Can the optimizer recommend it?

In [16]:
rows = []
for y in YEARS:
    G = eci.notebook_filter(R.lab_groups_iso3(y))
    e, p, d, M = eci.eci_from_exports(G, strict=True, return_all=True)
    pr = p.rank(ascending=False)
    rows.append({'year': y, 'PCI Services': p.get(SVC), 'PCI rank (1=most complex) of 18': int(pr[SVC]),
                 'ubiquity (# countries RCA>1)': int(M[SVC].sum()), 'countries': len(M),
                 'report countries with RCA>1 in Services': ' '.join([c for c in REPORT if c in M.index and M.loc[c, SVC] == 1])})
F = pd.DataFrame(rows).set_index('year'); F

,PCI Services,PCI rank (1=most complex) of 18,ubiquity (# countries RCA>1),countries,report countries with RCA>1 in Services
year,,,,,
2019,0.2051,8,19,120,SGP
2020,0.3950,7,22,120,CAN SGP
2021,0.5067,6,23,105,CAN SGP GBR
2022,1.6191,2,19,120,SGP GBR
2023,0.9737,4,25,122,CAN SGP


**Takeaway (F):** yes. The Services group's PCI rises from rank 8 of 18 (0.205, 2019) to **rank 2 (1.619, 2022)**, and 19–25 countries have RCA > 1 in it. Singapore has RCA > 1 every year, and Canada and the UK in some years. Because PCI drives opportunity gain and the greedy objective, the lab's SDV and budget recommendations can tell a country to grow "Services and Utilities". That really means growing exports its customs office doesn't classify.

In [17]:
print(f'runtime: {time.time() - T0:.1f} s')

runtime: 3.5 s


## Caveats

- **Grouped ECI is coarse.** Every grouped variant correlates 0.48–0.56 with income, against about 0.78 for HS4. The changes here are small compared with that gap, and with the 11-15/11-16 finding that the bootstrap SE of a one-country 2019→2022 change is about 0.15 SD even at HS4. The direction flips in E are on grouped ECI, where noise is larger still.
- **The "not services" conclusion rests on in-repo evidence:** the single HS6 code, the match to WDI *merchandise* totals, and the cleaning dict `"Services and Utilities": [99]`. I did not download Comtrade EB02 services data to show the gap directly. The UK's real services exports are far larger than $17.9B, but that number is not in the repo, so I don't quote one.
- **The 2019–2021 cleaned files differ slightly from today's raw files** (A1b, C0). V0 uses the cleaned files, as the reports do.
- **The UAE peer standing** rests on 1 peer under both versions.
- **2023 is suspect (D1).** It is shown but not used for the means.
- **Nominal values, one-shot cross-sections, and no decision threshold pre-stated** for dropping the column.

## Helpful to know

- `Services and Utilities` = HS6 `999999` = "commodities not specified according to kind". Also, code 99 sits **inside** the High-complexity aggregate (`[28, 29, 30, 38, 83, 84, 87, 92, 99]`), so 999999 is double-counted (in both columns). This is one more piece of D1's overlap.
- `Data_Retrieval_2025.ipynb` has a `get_service()` function that pulls Comtrade EB02 (type `S`) services, but its output is not in `data/raw` or the cleaned CSVs. The column comes only from goods code 999999.
- **Saudi Arabia's goods exports sit mostly in 999999** (80.4% in 2019). Every HS4 analysis here (`repo_hs`, `load_hs4`) drops chapters 98/99, so Saudi Arabia appears as a much smaller, non-oil exporter at HS4. This matches the Q13 note (`proposals/2026-11-12-all-questions.md` #6) that Saudi Arabia's 2019 pull is $49B against the Growth Lab's $189B. The USA's 999999 is $173.9B (10.6%).
- The Philippines (Vietnam peer) and Thailand drop out of the grouped ECI **only** because of this column. The Fall 2025 Singapore report's missing Thailand (Q01) comes from the same cause.
- Dropping a column never brings back Saudi Arabia in 2022 or Kuwait in 2021: they are not in the cleaned CSVs at all.

## New research avenues

Each fits one night (also in `proposals/2026-11-17.md`).

1. **SDV and greedy targets without the Services column.** For the 7 Spring 2026 countries, recompute opportunity gain (methodology §5) and the one-step 5% greedy target with V0 and V1. Count how often "Services and Utilities" is the top SDV or greedy pick under V0, and what replaces it under V1.
2. **Saudi Arabia and the other heavy 999999 reporters at HS4.** Measure how HS4 ECI for Saudi Arabia, Russia, Denmark and the USA changes if 999999 is kept as a pseudo-product, or if their totals are scaled to WDI merchandise exports. Then compare with ATLAS.
3. **Why dropping Energy Drilling and Mining in 2022 flips the eigenvector.** Look at the eigengap and the λ2/λ3 eigenvectors of the 17-column 2022 matrix. Test whether the grouped ECI is one column away from switching axis in other years (a fragility check for the grouped reports).
4. **The aggregate without code 99 (V2) loses income signal.** Test whether that is driven by a handful of large 999999 reporters (USA, Denmark, Finland) by dropping them one at a time.

## Data issues / data needed

- **Data issue:** the `Services and Utilities` column is HS6 999999 (unspecified goods), not services. Table 1 of every report and `docs/methodology.md` describe it wrongly. See `data-requests/2026-11-17-services-trade.md`.
- **Data needed (optional):** real services exports (Comtrade EB02 total services, code 200, or WTO–UNCTAD–ITC / BaTiS) for 2019–2023, if the lab wants a true services column.
- **Data issue:** Kuwait (2021) and Saudi Arabia (2022) are missing from the cleaned CSVs entirely. This removes UAE peers from the comparison.